<header style="padding:1px;background:#00b2b1;border-top:5px solid #E37C4D">

# ⚙️ EDA of a time series using pmdarima (Auto.ARIMA)

<div class="alert alert-block alert-info">
🎓 Automatic Model Selection and Multi-Step Forecasting.
</div>

In [ ]:
## Instantiate the class with the data and the target column
analyzer = ArimaTimeSeriesAnalyzer(data_series_df1, 'Penrose', 'PM2.5', is_auto_arima=False)

In [ ]:
analyzer.timeseries_visualization(lags=15, max_diff_order=1)

In [ ]:
print(f'✅ Step 3: If data is skewed, transform the data (using a Log or Box-Cox transformation) to stabilise the variance.')

# print("BoxCoxEndogTransformer or LogEndogTransformer...")
# ## Selecting the transformer based on the data characteristics
# if analyzer.y_train.min() > 0:
#     transformer = BoxCoxEndogTransformer(lmbda2=1e-6)  # Using a small lambda2 for numerical stability
# else:
#     transformer = LogEndogTransformer()  # Fallback to log if data contains zeros or negative values after handling

# ## Note: Ensure y_train does not contain zero or negative values if using LogEndogTransformer
# transformed_y_train = transform_and_diagnose(analyzer.y_train, transformer)

In [ ]:
print(f'✅ Step 4: Use Auto.ARIMA to automatically find the best ARIMA/SARIMA model')

analyzer.auto_arima_model(analyzer.y_train)

In [ ]:
models = [
    ('ARIMA(0,1,4)', (0, 1, 4), None),
    ('ARIMA(1,1,4)', (1, 1, 4), None),
    ('ARIMA(1,0,1)', (1, 0, 1), None),
    ('ARIMA(2,0,1)', (2, 0, 1), None),
    # ('SARIMA(0,1,2)(0,0,2)[24]', (0, 1, 2), (0, 0, 2, 24)),
]

## Note: analyzer.y_train and analyzer.y_test are properly defined
analyzer.model = analyzer.model_selection(models, analyzer.y_train, analyzer.y_test, preference='AIC')

if not analyzer.model:
    logging.info("Failed to determine the best model based on AIC.")
else:
    logging.info(f"Best model based on AIC = {analyzer.model.aic():.2f} \n {analyzer.model.summary()} ")

# analyzer.model

In [ ]:
print(f'✅ Step 5: Check the Residuals by plotting the ACF of the Residuals ...')

print(f'🎓 [5.1] [Default] 4 Model Diagnostics plots: The Standardized Residual, Histogram plus KDE estimate (N(0,1): the normal distribution), Normal Q-Q, and the Correlogram. ...')
analyzer.model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
if hasattr(analyzer.model, 'resid'):
    residuals = analyzer.model.resid   ## Get the residuals from an instance of a fitted ARIMA model
    plot_residuals(residuals)                ## Plotting the residuals 

In [ ]:
## [Future Work] SARIMA Models
# analyzer.auto_sarima_model(analyzer.y_train)

## ArimaTimeSeriesAnalyzer.model  is an instance of a fitted ARIMA model
## ArimaTimeSeriesAnalyzer.sarima_model is an instance of a fitted SARIMA model

# # print(analyzer.sarima_model.summary())    ## [DEBUG] Printing model summary
# ## [Precondition] `analyzer.sarima_model` exists and has a `resid` attribute
# if hasattr(analyzer, 'model') and hasattr(analyzer.sarima_model, 'resid'):
#     plot_residuals(analyzer.sarima_model.resid)
#     # residuals = analyzer.sarima_model.resid   ## Get the residuals from an instance of a fitted ARIMA model
#     # plot_residuals(residuals)                 ## Plotting the residuals    

# ## 4 Model Diagnostics plots: The Standardized Residual, Histogram plus KDE estimate (N(0,1): the normal distribution), Normal Q-Q, and the Correlogram.
# analyzer.sarima_model.plot_diagnostics(figsize=(15,12))
# plt.show()

In [ ]:
# analyzer.model_selection()

# analyzer.multi_step_forecast(n_periods=168)  ## Forecasting next 24*7 hours (1 week)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

def one_step_forecast(analyzer, test):
    """
    Create multi-step forecasts and plots them alongside the test data.
    """
    test_len = len(test)
    ## Create predictions for the future, evaluate on test
    fcast, conf_int = analyzer.model.predict(n_periods=test_len, return_conf_int=True, alpha=0.05)
    # fcast = analyzer.model.predict(n_periods=test_len, return_conf_int=True, alpha=0.05)
    # forecasts = fcast[0]
    # confidence_intervals = fcast[1]
    # confidence_intervals = np.asarray(fcast[1])

    ## Plotting
    plot_it(analyzer.train_data[analyzer.target_col], analyzer.test_data[analyzer.target_col], fcast, conf_int)

def one_period_forecast(model, n_periods=168):
    """
    Forecast multiple periods ahead with confidence intervals.

    Parameters:
    model (ARIMA): The ARIMA model used for forecasting.
    n_periods (int): Number of periods to forecast.

    Returns:
    tuple: A tuple containing forecasts and their confidence intervals.
    """
    ## Create predictions for the future, evaluate on test
    fcast, conf_int = model.predict(n_periods=n_periods, return_conf_int=True, alpha=0.05)
    return fcast.tolist(), np.asarray(conf_int).tolist()
    
    # fcast = model.predict(n_periods=n_periods, return_conf_int=True, alpha=0.05)
    ## fcast is a list of two lists.
    ## The first list is the forecast
    # forecasts = fcast[0].tolist()
    ## The second list is the confidence interval
    # confidence_intervals = fcast[1]
    # return ( forecasts, 
    #          np.asarray(confidence_intervals).tolist()[0])
    
def multi_step_forecast(analyzer, test, period='week'):
    """
    Generate multi-step forecasts and update the model with actual observations for each period/forecast.
    Also plots them alongside the test data.

    Parameters:
    - analyzer (TimeSeriesAnalyzer): An instance containing the ARIMA model and data.
    - test (pd.Series): Test data for the target column.
    - period (str): The period for grouping test data, 'week' for hourly data equals 168 hours.

    Uses dynamic forecasting where the model is updated after each new observation.
    Prints progress and debugging information.
    """
    ## Initialize lists to hold predictions and confidence intervals
    forecasts = []
    confidence_intervals = []

    ## Calculate the number of periods based on the test data length and the defined period length: One week in hours = 24 * 7
    # period_length = 168 if period == 'week' else len(test)  
    ## One week in hours = 24 * 7 = 168; also 24 hours per day
    period_length = 168 if period == 'week' else 24 if period == 'day' else len(test)

    ## Generate predictions and update the model for each period: Loop over the test data in chunks
    for start_idx in range(0, len(test), period_length):
        end_idx = min(start_idx + period_length, len(test))
        current_test = test.iloc[start_idx:end_idx]
        
        ## Perform forecasting for the current period
        fc, conf = one_period_forecast(analyzer.model, n_periods=len(current_test))
        forecasts.extend(fc)
        confidence_intervals.extend(conf)
        # confidence_intervals.append(conf)

        ## Adaptive Forecasting: Update the model with actual observations from the current period
        analyzer.model.update(current_test.values)
        logging.debug(f'Iteration from {start_idx} to {end_idx}: Model updated with actual data with Forecast={fc}, CI={conf}.')

    ## Plot the results: Plotting the forecasts against the actual data
    plot_it(analyzer.train_data[analyzer.target_col], test, forecasts, np.array(confidence_intervals))
 
    ## Plotting the forecasts against the actual data
    # plot_it(analyzer.train_data[analyzer.target_col], analyzer.test_data[analyzer.target_col], fcast, conf_int)
    # plot_it(analyzer.train_data[analyzer.target_col], analyzer.test_data[analyzer.target_col], np.array(forecasts), np.array(confidence_intervals))
    
    
# @staticmethod
def plot_it(train, test, forecasts, confidence_intervals):
    """
    Plot training data, test data, forecasts, and confidence intervals.
    
    Parameters:
    - train (pd.Series): Training data series.
    - test (pd.Series): Test data series.
    - forecasts (list): List of forecasted values.
    - confidence_intervals (array): Array of confidence interval bounds.
    """
    fig, ax = plt.subplots(figsize=(18, 10))

    ## Plot training data
    ax.plot(train.index, train, color='blue', label='Training Data')

    ## Plot actual test data
    ax.plot(test.index, test, color='green', label='Actual Test Data')
    
    ## Plotting Confidence Intervals
    # lower_bounds = [ci[0] for ci in confidence_intervals]
    # upper_bounds = [ci[1] for ci in confidence_intervals]
    ax.fill_between(test.index, confidence_intervals[:, 0], confidence_intervals[:, 1], color='orange', alpha=0.3, label='95% Confidence Interval')

    ## Plot forecasts
    ax.plot(test.index, forecasts, color='red', label='Forecasted Data', marker='o')
    
    ax.set_title(f'Multi-Step Forecast Evaluation: Forecast vs Actuals')
    ax.set_xlabel('Date')
    ax.set_ylabel(f'{train.name}')
    ax.legend()
    plt.grid(True)
    plt.show()

    ## Evaluate and print forecast accuracy
    mape = mean_absolute_percentage_error(test, forecasts)
    # print(f"MAPE: {mape:.2%}")
    # ## Evaluate and print forecast accuracy 
    # mape = mean_absolute_percentage_error(analyzer.test_data, forecasts)
    logging.info(f"MAPE for the forecast: {mape:.2%}")

In [ ]:
one_step_forecast(analyzer, analyzer.test_data[analyzer.target_col])

In [ ]:
multi_step_forecast(analyzer, analyzer.test_data[analyzer.target_col], period='week')

In [ ]:
multi_step_forecast(analyzer, analyzer.test_data[analyzer.target_col], period='day')

### WIP -->

In [ ]:
# multi_step_forecast(analyzer, analyzer.test_data[analyzer.target_col])

# Example Usage
# Assuming 'model', 'train_data', 'test_data', and 'target_column' are predefined
analyzer = TimeSeriesAnalyzer(model, train_data, test_data, 'PM2.5', update_model=True)
multi_step_forecast(analyzer, update_freq=168)

In [ ]:
def one_period_forecast(model):
    """
    Performs a one-step forecast using the current model and updates the model with actual observation.
    """
    ## Perform one-step forecast
    # fcast, conf_int = analyzer.model.get_forecast(steps=1).summary_frame(alpha=0.05)
    # fcast = model.predict(n_periods=1, return_conf_int=True, alpha=0.05)
    # fcast, conf_int = analyzer.model.predict(n_periods=1, return_conf_int=True, alpha=0.05)
    # fcast, conf_int = analyzer.model.predict(n_periods=test_len, return_conf_int=True, alpha=0.05)

    ## Perform one-step forecast with pmdarima's auto_arima model
    forecast, conf_int = model.predict(n_periods=1, return_conf_int=True, alpha=0.05)

    ## The forecast and conf_int are numpy arrays, convert them to lists
    forecast = forecast.tolist() # forecast is now a list
    conf_int = conf_int.tolist() # conf_int is now a list of lists [[lower, upper]]

    # # Convert forecast and confidence intervals to lists for easy manipulation
    # forecast = fcast['mean'].tolist()
    # confidence_interval = conf_int[['mean_ci_lower', 'mean_ci_upper']].values.tolist()[0]

    # return forecast, confidence_interval
    return forecast[0], conf_int[0] # Return the first (and only) forecast and its confidence interval


def multi_step_forecast(analyzer):
    """
    Performs multi-step forecasting by iterating over the test data and updating the model after each prediction.
    """
    forecasts = []
    confidence_intervals = []

    # Temporary store of the original model to reset after forecasting
    original_model = analyzer.model

    # Iterate over the test data for out-of-sample forecasts
    for new_obs in analyzer.test_data[analyzer.target_col]:
        forecast, conf_int = one_period_forecast(analyzer.model)
        forecasts.append(forecast)  # append instead of extend
        confidence_intervals.append(conf_int)

        # Update the model with the new actual observation
        # In pmdarima, you cannot update the model in place, so we refit the model with additional observation
        # This is computationally expensive and not the exact equivalent of updating the model
        analyzer.model = analyzer.model.update(new_obs)

    # Plot the forecast results
    plot_it(analyzer.train_data, analyzer.test_data, forecasts, confidence_intervals)

    # Reset the model to the original state before forecasting
    analyzer.model = original_model

# Call the function
multi_step_forecast(analyzer)


<div class="alert alert-block alert-info">
🎓 The above differencing, ACF, and PACF suggest ARIMA(1,1,4).
</div>

1. [Differencing] The original (adfuller <0.05 ?), 1st and 2nd differenced time series are stationary. Typically the **1st differencing** is enough. Differencing for a stationary time series still will be stationary.
   * When a time series is differenced, each data point represents the change from the previous time step. This can help stabilize the mean of the time series by removing changes in the level of a time series, and thus stabilizing variance and making the series more stationary.
3. [ACF to suggest the orders of MA]: A bar crossing the significance limit means it is statistically significant. The blue region is the significance limit (The alpha=0.05 setting specifies a 95% confidence level).
   * Lag 1 in the original line is significant. 
   * Lag 1 in the 1st differencing line is significant (?). This means the model is expected to have the lag 1 term and there is the 1st differencing.
4. [Use the PACF to suggest the orders of AR]

* ACF measures the correlation between a time series and its lagged values. It tells us how much the current value of the time series is related to its past values.
* PACF, on the other hand, measures the partial correlation between a time series and its lagged values, after accounting for the effects of all the lagged values that come before it. It helps us determine whether there is a direct relationship between the current value of the time series and a specific lagged value, after controlling for the effects of all the other lagged values.

* Dickey-Fuller Test: The p-value helps to formally test for stationarity:
  * Stationary: p-values less than 0.05 suggest the series does not have unit roots, indicating stationarity.
  * Non-Stationary: p-values greater than or equal to 0.05 suggest the presence of unit roots, indicating non-stationarity.

### WIP -->

In [ ]:
import numpy as np
import pandas as pd
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima.arima.utils import ndiffs
from pmdarima.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import mean_squared_error

class ArimaTimeSeriesAnalyzer:

    def d_differencing_stationarity(self, series_name, alpha=0.05):
        """
        Test for stationarity and perform differencing if necessary.
        """
        series = self.data[series_name]
        kpss_diffs = ndiffs(series, alpha=alpha, test='kpss', max_d=6)
        adf_diffs = ndiffs(series, alpha=alpha, test='adf', max_d=6)
        n_diffs = max(adf_diffs, kpss_diffs)
        print(f"Suggested differencing term: {n_diffs}")
        return n_diffs

    def plot_acf_pacf(self, series_name, lags=15):
        """
        Plot ACF and PACF for the series.
        """
        series = self.data[series_name].dropna()
        fig, axes = plt.subplots(1, 2, figsize=(16, 3))
        plot_acf(series, ax=axes[0], lags=lags)
        plot_pacf(series, ax=axes[1], lags=lags)
        plt.show()

    def model_selection(self, series_name):
        """
        Automatically find the best ARIMA model.
        """
        series = self.data[series_name].dropna()
        # Use train/test split for model selection
        train, test = train_test_split(series, test_size=0.2)
        self.model = pm.auto_arima(train, error_action='ignore', trace=1,
                                   suppress_warnings=True, maxiter=10,
                                   seasonal=False, m=24)
        # Validation
        forecast, conf_int = self.model.predict(n_periods=len(test), return_conf_int=True)
        plt.figure(figsize=(10, 5))
        plt.plot(train.index, train, label='Training')
        plt.plot(test.index, test, label='Test')
        plt.plot(test.index, forecast, label='Forecast')
        plt.fill_between(test.index, conf_int[:, 0], conf_int[:, 1], alpha=0.1)
        plt.legend()
        plt.show()
        print(f"Model AIC: {self.model.aic()}")
        return self.model

    def multi_step_forecast(self, steps):
        """
        Create multi-step forecasts for a specified number of future steps.
        """
        forecast, conf_int = self.model.predict(n_periods=steps, return_conf_int=True)
        return forecast, conf_int

    def update_model(self, new_obs):
        """
        Update the model with new observations.
        """
        self.model.update(new_obs)

    def calculate_mape(self, actual, predicted):
        """
        Calculate mean absolute percentage error.
        """
        return np.mean(np.abs(actual - predicted) / actual) * 100

# Assume `rawdata_site1` is your loaded dataset
data_series_df1 = pd.read_csv('path_to_file.csv', index_col='Timestamp', parse_dates=True)
analyzer = ArimaTimeSeriesAnalyzer(data_series_df1)

# Example usage
series_name = 'PM2.5'
analyzer.display_key_statistics(series_name)
n_diffs = analyzer.d_differencing_stationarity(series_name)
analyzer.plot_acf_pacf(series_name)
best_model = analyzer.model_selection(series_name)
forecasts, conf_int = analyzer.multi_step_forecast(48)  # Forecasting next 48 hours


In [ ]:
# from pmdarima import auto_arima, ARIMA
from pmdarima.model_selection import train_test_split
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import mean_absolute_percentage_error

class ArimaTimeSeriesAnalyzer:
    def __init__(self, data):
        self.data = data
        self.model = None
        self.forecasts = None
        self.confidence_intervals = None

    def display_key_statistics(self, series_name):
        series = self.data[series_name]
        plt.figure(figsize=(10, 6))
        plt.plot(series)
        plt.title(f'{series_name} Time Series')
        plt.show()
    
    def d_differencing_stationarity(self, series_name):
        series = self.data[series_name]
        result = adfuller(series.dropna())
        print(f'Dickey-Fuller Test: ADF Statistic: {result[0]} p-value: {result[1]}')
        for key, value in result[4].items():
            print('Critial Values:')
            print(f'   {key}, {value}')
    
    def plot_acf_pacf(self, series_name):
        series = self.data[series_name].dropna()
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
        plot_acf(series, ax=ax1, title=f'Autocorrelation for {series_name}')
        plot_pacf(series, ax=ax2, title=f'Partial Autocorrelation for {series_name}')
        plt.tight_layout()
        plt.show()
    
    def model_selection(self, series_name):
        series = self.data[series_name].dropna()
        self.model = pm.auto_arima(series, seasonal=True, m=24, 
                                stepwise=True, suppress_warnings=True, 
                                error_action="ignore", trace=True)
        print(self.model.summary())
        return self.model
    
    def multi_step_forecast(self, series_name, steps):
        train, test = train_test_split(self.data[series_name], train_size=int(len(self.data[series_name]) * 0.8))
        self.model.fit(train)
        self.forecasts, self.confidence_intervals = self.model.predict(n_periods=steps, return_conf_int=True)
        self.plot_forecasts(train, test, self.forecasts, self.confidence_intervals, series_name)
    
    def plot_forecasts(self, train, test, forecasts, intervals, series_name):
        plt.figure(figsize=(12, 5))
        plt.plot(train.index, train, label='Training')
        plt.plot(test.index[:len(forecasts)], forecasts, color='red', label='Predicted')
        plt.fill_between(test.index[:len(forecasts)], intervals[:, 0], intervals[:, 1], color='pink', alpha=0.3, label='Confidence Interval')
        plt.title(f'Forecast vs Actuals for {series_name}')
        plt.legend()
        plt.show()
    
    def update_model(self, new_obs):
        self.model.update(new_obs)
    
    @staticmethod
    def mean_absolute_percentage_error(y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Usage Example:
analyzer = ArimaTimeSeriesAnalyzer(data_series_df1)
analyzer.display_key_statistics('PM2.5')
analyzer.d_differencing_stationarity('PM2.5')
analyzer.plot_acf_pacf('PM2.5')
analyzer.model_selection('PM2.5')
analyzer.multi_step_forecast('PM2.5', 24)  # Forecast next 24 hours


In [ ]:

# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## TODO
# series: pd.Series

In [ ]:
class TeradataTimeSeriesAnalyzer(TimeSeriesAnalyzer):
    """
    Concrete implementation for time series analysis using teradataml.
    """
    ## Implement all abstract methods for Teradata environment

    # def analyze(self, series):
    #     import teradataml as tdml
    #     from teradataml.analytics.mle import AutoArima
    #     # Convert pandas Series to teradataml DataFrame if needed
    #     if isinstance(series, pd.Series):
    #         series = tdml.DataFrame.from_pandas(series.to_frame())
    #     # Implement analysis using AutoArima
    #     # model = tdml.XXX.AutoArima(series)
    #     model = AutoArima(series)
    #     # Add additional analysis steps as needed
    #     return model
    
    def cleanup(self):
        # Specific cleanup for Teradata environment
        pass

class ArimaTimeSeriesAnalyzer(TimeSeriesAnalyzer):
    """
    Concrete implementation for time series analysis using pmdarima/statsmodels.
    """
    ## Implement all abstract methods for pmdarima/statsmodels (auto.arima) environment

    # def analyze(self, series: pd.Series):
    #     # Implement analysis using pmdarima
    #     model = pm.auto_arima(series)
    #     # Add additional analysis steps as needed
    #     return model

    def d_differencing_stationarity(self, df, column):
        # Specific implementation using statsmodels
        result = adfuller(df[column].dropna())
        print(f'ADF Statistic: {result[0]}')
        print(f'p-value: {result[1]}')
    
    def calculate_acf_pacf(self, df, column):
        # Specific implementation using statsmodels
        lag_acf = acf(df[column].dropna(), nlags=15)
        lag_pacf = pacf(df[column].dropna(), nlags=15, method='ols')
        return lag_acf, lag_pacf
    
    def plot_acf_pacf(self, acf_array, pacf_array):
        # Specific implementation using plotly or matplotlib
        fig = go.Figure()
        fig.add_trace(go.Scatter(y=acf_array, name='ACF'))
        fig.add_trace(go.Scatter(y=pacf_array, name='PACF'))
        fig.show()
    
    def run_arima_model(self, df, column):
        # Specific implementation using statsmodels
        pass
    
     def get_data(self):
        # Implementation for getting data using pandas and other sources
        pass

    def analyze_raw_data(self):
        # Implementation for raw data analysis
        pass

    def d_differencing_stationarity(self):
        # Dickey Fuller Test implementation using statsmodels
        pass

    def autocorrelation_analysis(self):
        # ACF and PACF analysis using statsmodels
        pass

    def arima_modeling(self):
        # ARIMA modeling steps using statsmodels
        pass
    
    def cleanup(self):
        # Specific cleanup for local environment
        pass

class TimeSeriesAnalysisFactory:
    """
    Factory Design Pattern to create and return a time series analyzer based on the given environment.
    """
    @staticmethod
    def get_time_series_analyzer(environment, ):
        if environment == 'teradataml':
            loader = TeradataDataLoader()
            return TeradataTimeSeriesAnalyzer(loader)
        elif environment == 'auto.arima':
            loader = PandasDataLoader()
            return StatsmodelsTimeSeriesAnalyzer(loader)
        else:
            raise ValueError(f"Unknown environment: {environment}")


## Main function to run the analysis
def perform_ts_eda(analysis_impl: TeradataTimeSeriesAnalyzer):
    analysis_impl.get_data()
    analysis_impl.timeseries_visualization()
    analysis_impl.transform_data
    analysis_impl.ensure_stationarity
    analysis_impl.analyze_raw_data()
    analysis_impl.check_stationarity()
    analysis_impl.acf_pacf_analysis()
    analysis_impl.autocorrelation_analysis()
    analysis_impl.arima_modeling()
    analysis_impl.model_selection()
    analysis_impl.residual_analysis()
    analysis_impl.calculate_forecasts()
    analysis_impl.cleanup()

## This will create an analyzer object based on the given environment. Instantiate and run analysis for both environments.
# teradata_analyzer = TimeSeriesAnalysisFactory.get_time_series_analyzer('teradataml')
autoarima_analyzer = TimeSeriesAnalysisFactory.get_time_series_analyzer('auto.arima')

# perform_ts_eda(teradata_analyzer)
perform_ts_eda(autoarima_analyzer)

<header style="padding:1px;background:#00b2b1;border-top:5px solid #E37C4D">

## ⚙️ Decomposition and Stationarity

In [ ]:
## Decomposition and Stationarity
variables = {'PM2.5': 'Particulate Matter <2.5 µm', 'PM10': 'Particulate Matter <10 µm'}
seasonal_decomposition_and_stationarity(rawdata_site1, 'Penrose', variables, 'H', 'Daily')

In [ ]:
class TimeSeriesEDA:
    """
    Enhanced Time Series Analysis focusing on effective visualization and communication of air quality data results/insights.
    Note: The data should be in chronological order and the timestamps should be equidistant in time series.

    Initialization: The initialization method configures the dataframe, ensuring that the 'Timestamp' and 'Site' columns exist and are properly formatted.
    
    Seasonal Decomposition: This method performs seasonal decomposition for selected sites and visualises the components. It's critical for understanding underlying trends and seasonal patterns in air quality data.
    
    Stationarity Test: Performs ADF and KPSS tests to evaluate the stationarity of the time series, which is required for proper ARIMA modelling.
    """
    
    def __init__(self, series, timestamp_col='Timestamp'):
        """
        Initializes the class with a dataframe and sets the timestamp column as the datetime index.
        Assumes the dataframe has a column named as specified by `timestamp_col` which should be converted to datetime if not already.
        
        Parameters:
        - dataframe: A pandas DataFrame that includes a timestamp column.
        - timestamp_col: The name of the column in `dataframe` to set as the datetime index.
        """

        # self.df = pd.read_csv(filepath, parse_dates=['Timestamp'], index_col='Timestamp')
        self.df = series.copy()
        # if timestamp_col not in self.df.columns:
        #     raise ValueError(f"The dataframe does not contain a column named '{timestamp_col}'")
        # ## Convert the timestamp column to datetime if it's not already
        # if not pd.api.types.is_datetime64_any_dtype(self.df[timestamp_col]):
        #     self.df[timestamp_col] = pd.to_datetime(self.df[timestamp_col])
        
        ## Step 1. Prophet expects the dataset to have two columns: ds and y. 
        ##         The ds column should be of a date format and y the variable we wish to forecast.
        # series.rename(columns={'Timestamp': 'ds', 'PM10': 'y'}, inplace=True)
        self.df.rename(columns={'Timestamp': 'ds'}, inplace=True)

        ## Handle multiple sites with ordered timestamps with multi-level index (or hierarchical index)
        if 'ds' not in self.df.columns or 'Site' not in self.df.columns:
            raise ValueError("DataFrame must contain 'Timestamp' and 'Site' columns.")
        ## Sets a multi-level index using 'Timestamp'|'ds' and 'Site'. This is crucial for localized time-series analyses.
        self.df['ds'] = pd.to_datetime(self.df['ds'])
        self.df.set_index(['Site', 'ds'], inplace=True)
        
        # self.scale_features()


    def train_test_split_cutoff_date(self, cutoff_date='2022-01-31', is_debug=False):
        """
        Splits the data into training and testing sets based on a cutoff date for each site.
        Provides detailed information about the shape of each dataset and prints this information if debugging is enabled.
        """
        cutoff_date = pd.to_datetime(cutoff_date)
        training_data = {}
        testing_data = {}
        sites = self.df.index.get_level_values('Site').unique()

        ## TODO: using numerical_columns_S1, numerical_columns_S2
        ## Define the specific columns for each site if known; otherwise use all columns available
        site_columns = {
            'Penrose': ['AQI', 'PM10', 'PM2.5', 'SO2', 'NO', 'NO2', 'NOx', 'Wind_Speed', 'Wind_Dir', 'Air_Temp', 'Rel_Humidity'],
            'Takapuna': ['AQI', 'PM10', 'PM2.5', 'NO', 'NO2', 'NOx', 'Wind_Speed', 'Wind_Dir', 'Air_Temp', 'Rel_Humidity']
        }

        for site in sites:
            site_data = self.df.xs(site, level='Site')
            ## Ensure there is data for the site
            if site_data.empty:
                print(f"No data available for site: {site}. Skipping...")
                continue
            ## Filter data to include only relevant columns for the site
            relevant_columns = site_columns.get(site, site_data.columns)  ## Default to all columns if site not specified
            site_data = site_data[relevant_columns].copy()                ## Ensure only relevant columns are considered

            ## Split data based on the cutoff date
            train = site_data[site_data.index <= cutoff_date]
            test = site_data[site_data.index > cutoff_date]

            ## Store training and testing data in dictionaries
            training_data[site] = train
            testing_data[site] = test

            ## Print information about the splits
            if is_debug:
                print(f"Site: {site}")
                print(f"Series shape: {site_data.shape}, Train DataFrame shape: {train.shape}, Test DataFrame shape: {test.shape}")
                if not train.empty:
                    print(f"Training data from {train.index.min()} to {train.index.max()}")
                if not test.empty:
                    print(f"Testing data from {test.index.min()} to {test.index.max()} \n")

        return training_data, testing_data

  
    def seasonal_decompose(self, target, model='additive', extrapolate_trend='freq', period=24):
        """
        Performs and visualizes seasonal decomposition of the target variable for specified sites.
        Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'.
        
        Parameters:
            target (str): The target variable column name for decomposition (e.g., 'PM2.5').
            model (str): The type of decomposition model ('additive' or 'multiplicative').
            period (int): The periodicity of the time series data (default is 24 for hourly data).
        """
        print(f"Seasonal Decomposition: Analyzes and plots seasonal decomposition of the {target} target variable.")
        sites = ['Penrose', 'Takapuna']
        fig, axes = plt.subplots(nrows=3, ncols=len(sites), figsize=(12, 12), sharex=True)
        
        for i, site in enumerate(sites):
            ## Extract site-specific data
            site_data = self.df.xs(site, level='Site')[target].dropna()
            if len(site_data) < 2 * period:
                raise ValueError(f"Not enough data points for seasonal decomposition at {site} site with period {period}.")

            # ## Check sufficient data for decomposition
            # if len(site_data) < 2 * period:
            #     raise ValueError(f"Not enough data points for seasonal decomposition at {site} site with period {period}.")
            ## Seasonal decomposition
            decomposition = sm.tsa.seasonal_decompose(site_data, model=model, extrapolate_trend=extrapolate_trend, period=period)
            components = [decomposition.trend, decomposition.seasonal, decomposition.resid]
            component_names = ['Trend', 'Seasonal', 'Residual']

            for j, component in enumerate(components):
                axes[j, i].plot(component, label=f'{site} {component_names[j]}')
                axes[j, i].set_title(f'{component_names[j]} for {site}')
                axes[j, i].legend()

        plt.tight_layout()
        plt.show()


    def stationarity_test(self):
        """
        Performs the Augmented Dickey-Fuller (ADF) and Kwiatkowski-Phillips-Schmidt-Shin (KPSS) stationarity tests on numeric columns across different sites to assess stationarity
        and outputs detailed results in DataFrame format, including hypothesis evaluation.
        [ADF Test][Null Hypothesis H0]: The series has a unit root (non-stationary).
        [KPSS Test][Null Hypothesis H0]: The series is stationary or trend-stationary.

        
        ADF tests for unit roots, while KPSS tests for stationarity around a trend.

        Returns a structured DataFrame with test results and logs any issues with data directly to the console.
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        sites = ['Penrose', 'Takapuna']
        results = []

        for site in sites:
            site_data = self.df.xs(site, level='Site')
            for column in numeric_cols:
                series_data = site_data[column].dropna()
                if series_data.empty:
                    print(f"Warning: No data available for {column} at {site} after dropna(). Skipping...")
                    continue

                ## Adjusters Dickey-Fuller (ADF) Test
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    adf_test_result = adfuller(series_data, autolag='AIC')
                
                ## Kwiatkowski — Phillips — Schmidt — Shin (KPSS) Test
                ## The KPSS test can also be used to detect stochastic trends.
                ## The test hypotheses are opposite relative to ADF:
                ## Null hypothesis: the time series is trend-stationary.
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    kpss_test_result = kpss(series_data, regression='ct')

                ## Prepare results for both tests including hypothesis interpretation
                results.append({
                    'Site': site,
                    'Variable': column,
                    'ADF Statistic': adf_test_result[0],
                    'ADF p-value': adf_test_result[1],
                    'ADF Critical Values': adf_test_result[4],
                    'Used Lag': adf_test_result[2],
                    # 'Observations': adf_test_result[3],
                    # 'Critical Value (1%)': adf_test_result[4]['1%'],
                    'Critical Value (5%)': adf_test_result[4]['5%'],
                    # 'Critical Value (10%)': adf_test_result[4]['10%'],
                    
                    'KPSS Statistic': kpss_test_result[0],
                    'KPSS p-value': kpss_test_result[1],
                    'KPSS Critical Values': kpss_test_result[3],
                    'Lags Used': kpss_test_result[2],
                    # 'Critical Value (1%)': kpss_test_result[3]['1%'],
                    'Critical Value (5%)': kpss_test_result[3]['5%'],
                    # 'Critical Value (10%)': kpss_test_result[3]['10%'],
                    ## ADF Test: Non-stationary if test statistic is greater than or equal to the 5% critical value
                    'ADF Conclusion': "Stationary" if adf_test_result[0] < adf_test_result[4]['5%'] else "Non-stationary",
                    ## KPSS Test: Non-stationary if test statistic is greater than or equal to the 5% critical value
                    'KPSS Conclusion': "Non-stationary" if kpss_test_result[0] >= kpss_test_result[3]['5%'] else "Stationary",
                })

        ## Convert results to DataFrame for better accessibility and display
        result_df = pd.DataFrame(results)
        return result_df

In [ ]:
## Filter rawdata for the specified sites
## [Note] Unsure impute missing values first
## Copy the rawdata for safety so the original data or prepared data from Teradata Vantage is not modified
if IS_DATA_IN_TERADATA_VANTAGE:
    series = df_table.to_pandas()
else:
    series = rawdata.copy()
timeseriesEDA = TimeSeriesEDA(series, timestamp_col='Timestamp')

## Additive model; period: 12 months in monthly dataset | daily dataset would be period=365 | hourly dataset would be 24.
print("\n[PM2.5] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'\n")
timeseriesEDA.seasonal_decompose('PM2.5', model='additive', extrapolate_trend='freq', period=24)
print("[PM10] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'")
timeseriesEDA.seasonal_decompose('PM10', model='additive', extrapolate_trend='freq', period=24)

## ADF & KPSS test results
stationarity_results = timeseriesEDA.stationarity_test()
stationarity_results

In [ ]:

## Additive model; period: 12 months in monthly dataset | daily dataset would be period=365 | hourly dataset would be 24.
print("\n[PM2.5] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'\n")
timeseriesEDA.seasonal_decompose('PM2.5', model='additive', extrapolate_trend='freq', period=24)
print("[PM10] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'")
timeseriesEDA.seasonal_decompose('PM10', model='additive', extrapolate_trend='freq', period=24)

timeseries_analysis = AdvancedTimeSeriesAnalysis(series, timestamp_col='Timestamp')
## Additive model; period: 12 months in monthly dataset | daily dataset would be period=365 | hourly dataset would be 24.
print("\n[PM2.5] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'\n")
timeseries_analysis.seasonal_decompose('PM2.5', model='additive', extrapolate_trend='freq', period=24)
print("[PM10] Plots the trend, seasonal, and residual components side by side for 'Penrose' and 'Takapuna'")
timeseries_analysis.seasonal_decompose('PM10', model='additive', extrapolate_trend='freq', period=24)

## ADF & KPSS test results
stationarity_results = timeseries_analysis.stationarity_test()
stationarity_results

In [ ]:
training_data, testing_data = timeseries_analysis.train_test_split_cutoff_date(cutoff_date='2021-12-31', is_debug=True)

## Retrieve training data for 'Penrose' & 'Takapuna'
training_data1 = training_data['Penrose']
training_data2 = training_data['Takapuna']

## Retrieve testing data for 'Penrose' & 'Takapuna'
testing_data1 = testing_data['Penrose']
testing_data2 = testing_data['Takapuna']

print(f"Series 1 shape - Penrose: {rawdata_site1.shape}, Train DataFrame shape: {training_data1.shape}, Test DataFrame shape: {testing_data1.shape}")
print(f"Series 2 shape - Takapuna: {rawdata_site2.shape}, Train DataFrame shape: {training_data2.shape}, Test DataFrame shape: {testing_data2.shape}")

## TODO
# model, predictions = some_model_training_function(...)
# analysis.plot_forecast_vs_actual(analysis.df['PM10'], predictions)
# analysis.plot_feature_importance(model, analysis.df.columns)

<header style="padding:1px;background:#00b2b1;border-top:5px solid #E37C4D">

# ⚙️ EDA of a Time Series using TeradataML in Teradata Vantage